# RAG 探索遊樂場 (exploration)

互動測試 abby-notes-rag 的檢索品質。

**用法**：從上到下，每格按 `Shift+Enter` 執行。
- 「載入」那格會載 bge-m3 model（約 10 秒），**只需跑一次**。
- 之後改 query / top_k 重跑該格即可，model 不用重載，秒回。

前置：Docker 的 `abby-rag-postgres` 容器要在跑（`docker compose up -d`），且已經 ingest 過。

## 0. 選用套件（表格 + 圖表）

`pandas`（漂亮表格）和 `matplotlib`（長條圖）預設沒裝。
要用第 2 節的表格和第 5 節的圖，先執行下面這格安裝；只想看純文字結果可略過。

In [ ]:
# 選用：安裝表格/圖表套件（已裝可略過）
%pip install -q pandas matplotlib

In [ ]:
import os
import sys
from pathlib import Path

# 切到專案根目錄（notebooks/ 的上一層），讓 src import 和 .env 都正常
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.retriever import Retriever

# 載入一次：bge-m3 model + pgvector 連線（約 10 秒，只跑這一次）
r = Retriever()

## 1. 基本查詢

回傳 top_k 個最相關的 chunk，印出相似度、檔案、標題路徑。

In [ ]:
results = r.search("ECPay 退款怎麼處理", top_k=5)
for x in results:
    print(f"{x['similarity']:.3f}  {x['file_path']}")
    print(f"        {x['heading_path']}")

## 2. 排成漂亮表格

`show()` 有 pandas 就回傳 DataFrame（表格），沒裝就退化成純文字。

In [ ]:
def show(results):
    """把 search 結果排成表格。有 pandas 用 DataFrame，沒有就純文字。"""
    try:
        import pandas as pd
        df = pd.DataFrame(results)[["similarity", "file_path", "heading_path"]]
        df["similarity"] = df["similarity"].round(3)
        return df
    except ImportError:
        for x in results:
            print(f"{x['similarity']:.3f}  {x['file_path']}")

show(r.search("Docker 遷移失敗 AutoMigrate", top_k=5))

## 3. 只搜某個資料夾（filter）

`filter_path_prefix` 限定只搜某前綴底下的檔案。

In [ ]:
# 只搜 RAG/ 資料夾底下的筆記
show(r.search("chunking 怎麼切", top_k=5, filter_path_prefix="RAG/"))

## 4. 調 top_k（model 不用重載，秒回）

改數字重跑這格，立刻看到更多/更少結果。

In [ ]:
show(r.search("Redis 資料型別", top_k=10))

## 5. 相似度長條圖（需要 matplotlib）

把 top_k 的相似度畫成長條，一眼看出「第一名跟後面差多少」。

In [ ]:
try:
    import matplotlib.pyplot as plt

    q = "booth 訂單規則"
    res = r.search(q, top_k=8)
    labels = [x["file_path"].split("/")[-1] for x in res]
    scores = [x["similarity"] for x in res]

    plt.figure(figsize=(8, 4))
    plt.barh(labels[::-1], scores[::-1])
    plt.xlabel("similarity")
    plt.title(f"Query: {q}")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("沒裝 matplotlib，請先執行第 0 節的安裝格")

## 6. 收尾

用完可關掉 DB 連線（非必要，kernel 關掉也會釋放）。

In [ ]:
# r.close()  # 取消註解即可關閉 DB 連線